# Modélisation 

### Import des modules 

In [57]:
import pandas as pd
import numpy as np

#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
import category_encoders as cat_encod
from sklearn.preprocessing import LabelEncoder,  OneHotEncoder, StandardScaler, MinMaxScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor


### Importation du fichier CSV 'ready_to_fe'
#### ready to feature engineering

In [58]:
building_consumption_ready_to_fe = pd.read_csv("data/Processed/building_consumption_ready_to_fe.csv")

In [59]:
building_consumption_ready_to_fe

,Unnamed: 0,Neighborhood,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,SiteEUI(kBtu/sf),SiteEnergyUse(kBtu),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,FE1_Is_multi_type
0,0,DOWNTOWN,1927,1.0,12,88434,0,88434,Hotel,81.699997,7.226362e+06,2003882.00,3.946027e+06,1.276453e+06,249.98,0
1,1,DOWNTOWN,1996,1.0,11,103566,15064,88502,"Hotel, Parking, Restaurant",94.800003,8.387933e+06,0.00,3.242851e+06,5.145082e+06,295.86,1
2,2,DOWNTOWN,1969,1.0,41,956110,196718,759392,Hotel,96.000000,7.258702e+07,21566554.00,4.952666e+07,1.493800e+06,2089.28,0
3,3,DOWNTOWN,1926,1.0,10,61320,0,61320,Hotel,110.800003,6.794584e+06,2214446.25,2.768924e+06,1.811213e+06,286.43,0
4,4,DOWNTOWN,1980,1.0,18,175580,62000,113580,"Hotel, Parking, Swimming Pool",114.800003,1.417261e+07,0.00,5.368607e+06,8.803998e+06,505.01,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,3371,GREATER DUWAMISH,1990,1.0,1,12294,0,12294,Office,69.099998,8.497457e+05,0.00,5.242709e+05,3.254750e+05,20.94,0
1588,3372,DOWNTOWN,2004,1.0,1,16000,0,16000,Other - Recreation,59.400002,9.502762e+05,0.00,3.965461e+05,5.537300e+05,32.17,0
1589,3373,MAGNOLIA QUEEN ANNE,1974,1.0,1,13157,0,13157,"Fitness Center/Health Club/Gym, Other - Recrea...",438.200012,5.765898e+06,0.00,1.792159e+06,3.973739e+06,223.54,1
1590,3374,GREATER DUWAMISH,1989,1.0,1,14101,0,14101,"Fitness Center/Health Club/Gym, Food Service, ...",51.000000,7.194712e+05,0.00,3.488702e+05,3.706010e+05,22.11,1


### Feature Engineering

A réaliser : Enrichir le jeu de données actuel avec de nouvelles features issues de celles existantes. 

En règle générale : On utilise la méthode .apply() de Pandas pour créer une nouvelle colonne à partir d'une colonne existante. N'hésitez pas à regarder les exemples dans les chapitres de cours donnés en ressource

#### Les nouvelles features : 
<table border="1" style="width: 100%; border-collapse: collapse;">
        <thead>
          <tr>
            <th style="padding: 8px; text-align: left;">Code</th>
            <th style="padding: 8px; text-align: left;">Nom de la feature</th>
            <th style="padding: 8px; text-align: left;">Ce que cette feature représente</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td style="padding: 6px;">FE1</td>
            <td style="padding: 6px;">FE1_is_multi_type</td>
            <td style="padding: 6px;">Permet de savoir si un bâtiment est multi usage ou non</td>
          </tr>
          <tr>
            <td style="padding: 6px;">FE2</td>
            <td style="padding: 6px;">FE2_year_in_activity</td>
            <td style="padding: 6px;">Calculer l'age des bâtiments (permet de savoir si un vieux bâtiment a tendance a consommer davantage)</td>
          </tr>
          <tr>
            <td style="padding: 6px;">FE3</td>
            <td style="padding: 6px;">FE3_floor_area</td>
            <td style="padding: 6px;">Permet de connaitre la superficie d'un étage (indication sur la consommation)</td>
          </tr>
        </tbody>
</table>

La feature FE1 a déja été créé durant l'analyse de données

In [60]:
# Code pour la FE2 
# L'année de référence est 2016 au lieu de 2026 car l'étude des bâtiments date de 2016 
building_consumption_ready_to_fe['FE2_year_in_activity'] = 2016 - building_consumption_ready_to_fe['YearBuilt']
building_consumption_ready_to_fe

,Unnamed: 0,Neighborhood,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,SiteEUI(kBtu/sf),SiteEnergyUse(kBtu),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,FE1_Is_multi_type,FE2_year_in_activity
0,0,DOWNTOWN,1927,1.0,12,88434,0,88434,Hotel,81.699997,7.226362e+06,2003882.00,3.946027e+06,1.276453e+06,249.98,0,89
1,1,DOWNTOWN,1996,1.0,11,103566,15064,88502,"Hotel, Parking, Restaurant",94.800003,8.387933e+06,0.00,3.242851e+06,5.145082e+06,295.86,1,20
2,2,DOWNTOWN,1969,1.0,41,956110,196718,759392,Hotel,96.000000,7.258702e+07,21566554.00,4.952666e+07,1.493800e+06,2089.28,0,47
3,3,DOWNTOWN,1926,1.0,10,61320,0,61320,Hotel,110.800003,6.794584e+06,2214446.25,2.768924e+06,1.811213e+06,286.43,0,90
4,4,DOWNTOWN,1980,1.0,18,175580,62000,113580,"Hotel, Parking, Swimming Pool",114.800003,1.417261e+07,0.00,5.368607e+06,8.803998e+06,505.01,1,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,3371,GREATER DUWAMISH,1990,1.0,1,12294,0,12294,Office,69.099998,8.497457e+05,0.00,5.242709e+05,3.254750e+05,20.94,0,26
1588,3372,DOWNTOWN,2004,1.0,1,16000,0,16000,Other - Recreation,59.400002,9.502762e+05,0.00,3.965461e+05,5.537300e+05,32.17,0,12
1589,3373,MAGNOLIA QUEEN ANNE,1974,1.0,1,13157,0,13157,"Fitness Center/Health Club/Gym, Other - Recrea...",438.200012,5.765898e+06,0.00,1.792159e+06,3.973739e+06,223.54,1,42
1590,3374,GREATER DUWAMISH,1989,1.0,1,14101,0,14101,"Fitness Center/Health Club/Gym, Food Service, ...",51.000000,7.194712e+05,0.00,3.488702e+05,3.706010e+05,22.11,1,27


In [61]:
# Code pour la FE3
building_consumption_ready_to_fe['FE3_floor_area'] = round(building_consumption_ready_to_fe['PropertyGFABuilding(s)'] / building_consumption_ready_to_fe['NumberofFloors'], 0)
building_consumption_ready_to_fe

,Unnamed: 0,Neighborhood,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,SiteEUI(kBtu/sf),SiteEnergyUse(kBtu),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,FE1_Is_multi_type,FE2_year_in_activity,FE3_floor_area
0,0,DOWNTOWN,1927,1.0,12,88434,0,88434,Hotel,81.699997,7.226362e+06,2003882.00,3.946027e+06,1.276453e+06,249.98,0,89,7370.0
1,1,DOWNTOWN,1996,1.0,11,103566,15064,88502,"Hotel, Parking, Restaurant",94.800003,8.387933e+06,0.00,3.242851e+06,5.145082e+06,295.86,1,20,8046.0
2,2,DOWNTOWN,1969,1.0,41,956110,196718,759392,Hotel,96.000000,7.258702e+07,21566554.00,4.952666e+07,1.493800e+06,2089.28,0,47,18522.0
3,3,DOWNTOWN,1926,1.0,10,61320,0,61320,Hotel,110.800003,6.794584e+06,2214446.25,2.768924e+06,1.811213e+06,286.43,0,90,6132.0
4,4,DOWNTOWN,1980,1.0,18,175580,62000,113580,"Hotel, Parking, Swimming Pool",114.800003,1.417261e+07,0.00,5.368607e+06,8.803998e+06,505.01,1,36,6310.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,3371,GREATER DUWAMISH,1990,1.0,1,12294,0,12294,Office,69.099998,8.497457e+05,0.00,5.242709e+05,3.254750e+05,20.94,0,26,12294.0
1588,3372,DOWNTOWN,2004,1.0,1,16000,0,16000,Other - Recreation,59.400002,9.502762e+05,0.00,3.965461e+05,5.537300e+05,32.17,0,12,16000.0
1589,3373,MAGNOLIA QUEEN ANNE,1974,1.0,1,13157,0,13157,"Fitness Center/Health Club/Gym, Other - Recrea...",438.200012,5.765898e+06,0.00,1.792159e+06,3.973739e+06,223.54,1,42,13157.0
1590,3374,GREATER DUWAMISH,1989,1.0,1,14101,0,14101,"Fitness Center/Health Club/Gym, Food Service, ...",51.000000,7.194712e+05,0.00,3.488702e+05,3.706010e+05,22.11,1,27,14101.0


### Préparation des features pour la modélisation

A réaliser :
* Si ce n'est pas déjà fait, supprimer toutes les colonnes peu pertinentes pour la modélisation.
* Tracer la distribution de la cible pour vous familiariser avec l'ordre de grandeur. En cas d'outliers, mettez en place une démarche pour les supprimer.
* Débarrassez-vous des features redondantes en utilisant une matrice de corrélation de Pearson. Pour cela, utiisez la méthode corr() de Pandas, couplé d'un graphique Heatmap de la librairie Seaborn 
* Réalisez différents graphiques pour comprendre le lien entre vos features et la target (boxplots, scatterplots, pairplot si votre nombre de features numériques n'est pas très élevé).
*  Séparez votre jeu de données en un Pandas DataFrame X (ensemble de feautures) et Pandas Series y (votre target).
* Si vous avez des features catégorielles, il faut les encoder pour que votre modèle fonctionne. Les deux méthodes d'encodage à connaitre sont le OneHotEncoder et le LabelEncoder

#### Suppression des Colonnes ListOfAllPropertyUseTypes, YearBuilt et Unnamed: 0 qui ne sont plus utiles

In [62]:
building_consumption_ready_to_fe.drop(columns=['ListOfAllPropertyUseTypes'], inplace=True)
building_consumption_ready_to_fe.drop(columns=['Unnamed: 0'], inplace=True)
building_consumption_ready_to_fe.drop(columns=['YearBuilt'], inplace=True)

#### Encodage de la variable Neighborhood avec la methode Category_Encoders
L'idée est de numériser les catégories pour faciliter le ML.   
Crée 4 colonnes Neighborhood (1, 2, 3 ,4) qui prennent pour valeur 0 ou 1 pour que chaque catégorie (quartier) soit encodée en binaire.   
DOWNTOWN devient 0001    
MAGNOLIA QUEEN ANNE devient 0110

In [63]:
encoder = cat_encod.BinaryEncoder(cols=["Neighborhood"])
building_consumption_encoded = encoder.fit_transform(building_consumption_ready_to_fe)

#### Avant la normalisation 

In [64]:
building_consumption_encoded

,Neighborhood_0,Neighborhood_1,Neighborhood_2,Neighborhood_3,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),SiteEUI(kBtu/sf),SiteEnergyUse(kBtu),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,FE1_Is_multi_type,FE2_year_in_activity,FE3_floor_area
0,0,0,0,1,1.0,12,88434,0,88434,81.699997,7.226362e+06,2003882.00,3.946027e+06,1.276453e+06,249.98,0,89,7370.0
1,0,0,0,1,1.0,11,103566,15064,88502,94.800003,8.387933e+06,0.00,3.242851e+06,5.145082e+06,295.86,1,20,8046.0
2,0,0,0,1,1.0,41,956110,196718,759392,96.000000,7.258702e+07,21566554.00,4.952666e+07,1.493800e+06,2089.28,0,47,18522.0
3,0,0,0,1,1.0,10,61320,0,61320,110.800003,6.794584e+06,2214446.25,2.768924e+06,1.811213e+06,286.43,0,90,6132.0
4,0,0,0,1,1.0,18,175580,62000,113580,114.800003,1.417261e+07,0.00,5.368607e+06,8.803998e+06,505.01,1,36,6310.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,1,0,0,0,1.0,1,12294,0,12294,69.099998,8.497457e+05,0.00,5.242709e+05,3.254750e+05,20.94,0,26,12294.0
1588,0,0,0,1,1.0,1,16000,0,16000,59.400002,9.502762e+05,0.00,3.965461e+05,5.537300e+05,32.17,0,12,16000.0
1589,0,1,1,0,1.0,1,13157,0,13157,438.200012,5.765898e+06,0.00,1.792159e+06,3.973739e+06,223.54,1,42,13157.0
1590,1,0,0,0,1.0,1,14101,0,14101,51.000000,7.194712e+05,0.00,3.488702e+05,3.706010e+05,22.11,1,27,14101.0


Verifier qu'il ne reste aucun NaN dans les colonnes a normaliser.   
Ne pas faire la normalisation sur la cible y.   
Il faut spliter en Train/Test car nous ne normalisons pas les X_test et Y_test.   

In [65]:
# création des Variables et Cible
X = building_consumption_encoded.drop(columns=['SiteEnergyUse(kBtu)'])
y = building_consumption_encoded['SiteEnergyUse(kBtu)']

# split (jeux de train et jeux de test)
# le ratio choisi est 80% Train et 20% Test
# le random_state est la seed pour les jeux de test (si besoin de reproduire exactement le meme jeu de test utiliser la meme seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_train = X_train.copy()
X_test = X_test.copy()

#### Normalisation des Features.  
Le but est de réduire les amplitudes entre les variables (Facilite le ML).   
* SteamUse(kBtu).   
* Electricity(kBtu).	   
* NaturalGas(kBtu).  
* PropertyGFABuilding(s).   
* PropertyGFAParking.   
* PropertyGFATotal.  
* SiteEUI(kBtu/sf).   

In [66]:
colonnes_a_normaliser = ['NumberofBuildings', 'NumberofFloors',
                         'PropertyGFATotal', 'PropertyGFAParking',
                         'PropertyGFABuilding(s)', 'SiteEUI(kBtu/sf)', 
                         'SteamUse(kBtu)', 'Electricity(kBtu)', 'NaturalGas(kBtu)', 
                         'TotalGHGEmissions', 'FE2_year_in_activity', 'FE3_floor_area']


# Création d'on objet de la classe MinMaxScaler
scaler = MinMaxScaler()

X_train[colonnes_a_normaliser] = scaler.fit_transform(X_train[colonnes_a_normaliser])
X_test[colonnes_a_normaliser] = scaler.transform(X_test[colonnes_a_normaliser])

In [67]:
X_train

,Neighborhood_0,Neighborhood_1,Neighborhood_2,Neighborhood_3,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),SiteEUI(kBtu/sf),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,FE1_Is_multi_type,FE2_year_in_activity,FE3_floor_area
1545,1,0,0,0,0.0,0.061224,0.147314,0.286158,0.083498,0.027828,0.0,0.015878,0.000747,0.002494,1,0.008696,0.024092
1509,1,0,0,0,0.0,0.030612,0.028132,0.000000,0.031517,0.023869,0.0,0.005302,0.000144,0.000730,0,0.965217,0.016361
744,1,0,0,0,0.0,0.000000,0.012208,0.000000,0.015648,0.022910,0.0,0.002305,0.000401,0.000630,1,0.469565,0.034355
462,0,0,0,1,0.0,0.051020,0.039491,0.000000,0.042836,0.039583,0.0,0.009026,0.000000,0.001017,1,0.773913,0.014607
1356,1,0,1,0,0.0,0.010204,0.020418,0.000000,0.023829,0.130623,0.0,0.009681,0.011673,0.012038,0,0.469565,0.025246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130,1,0,0,0,0.0,0.000000,0.005971,0.000000,0.009433,0.035384,0.0,0.001720,0.000885,0.001018,1,0.426087,0.021943
1294,0,0,0,1,0.0,0.040816,0.012378,0.000000,0.015817,0.077126,0.0,0.007498,0.000000,0.000843,1,1.000000,0.006777
860,0,1,1,0,0.0,0.000000,0.008794,0.000000,0.012246,0.246492,0.0,0.009583,0.010984,0.011381,0,0.147826,0.027561
1459,1,0,1,1,0.0,0.000000,0.045008,0.000000,0.048334,0.035145,0.0,0.004716,0.006719,0.006830,0,0.565217,0.099631


In [68]:
# Les colonnes fuite concernent les colonnes avec une relation forte avec la cible 
# Il faut les enlever pour eviter le surapprentissage
colonnes_fuite = ['TotalGHGEmissions', 'NaturalGas(kBtu)', 'SteamUse(kBtu)',
                  'Electricity(kBtu)', 'SiteEUI(kBtu/sf)']

X_train = X_train.drop(columns=colonnes_fuite)
X_test = X_test.drop(columns=colonnes_fuite)

Transformation de la variable cible en logatithme pour lisser les résultats

In [69]:
# Transformation de la variable cible en logarithme 
# Cela aidera a avoir des résultats plus cohérents sur les 3 modeles (notemment le SVR)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

Verification que les données sont 'saines'.   
Les 2 lignes doivent donner 0

In [70]:
# nombre de valeurs infinies 
print(np.isinf(X_train.select_dtypes("number")).sum().sum())

# nombre de vides 
print(X_train.isna().sum().sum()) 

0
0


In [71]:
X_train

,Neighborhood_0,Neighborhood_1,Neighborhood_2,Neighborhood_3,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),FE1_Is_multi_type,FE2_year_in_activity,FE3_floor_area
1545,1,0,0,0,0.0,0.061224,0.147314,0.286158,0.083498,1,0.008696,0.024092
1509,1,0,0,0,0.0,0.030612,0.028132,0.000000,0.031517,0,0.965217,0.016361
744,1,0,0,0,0.0,0.000000,0.012208,0.000000,0.015648,1,0.469565,0.034355
462,0,0,0,1,0.0,0.051020,0.039491,0.000000,0.042836,1,0.773913,0.014607
1356,1,0,1,0,0.0,0.010204,0.020418,0.000000,0.023829,0,0.469565,0.025246
...,...,...,...,...,...,...,...,...,...,...,...,...
1130,1,0,0,0,0.0,0.000000,0.005971,0.000000,0.009433,1,0.426087,0.021943
1294,0,0,0,1,0.0,0.040816,0.012378,0.000000,0.015817,1,1.000000,0.006777
860,0,1,1,0,0.0,0.000000,0.008794,0.000000,0.012246,0,0.147826,0.027561
1459,1,0,1,1,0.0,0.000000,0.045008,0.000000,0.048334,0,0.565217,0.099631


### Comparaison de différents modèles supervisés

A réaliser :
* Pour chaque algorithme que vous allez tester, vous devez :
    * Réaliser au préalable une séparation en jeu d'apprentissage et jeu de test via une validation croisée.
    * Si les features quantitatives que vous souhaitez utiliser ont des ordres de grandeur très différents les uns des autres, et que vous utilisez un algorithme de regression qui est sensible à cette différence, alors il faut réaliser un scaling (normalisation) de la donnée au préalable.
    * Entrainer le modèle sur le jeu de Train
    * Prédire la cible sur la donnée de test (nous appelons cette étape, l'inférence).
    * Calculer les métriques de performance R2, MAE et RMSE sur le jeu de train et de test.
    * Interpréter les résultats pour juger de la fiabilité de l'algorithme.
* Vous pouvez choisir par exemple de tester un modèle linéaire, un modèle à base d'arbres et un modèle de type SVM
* Déterminer le modèle le plus performant parmi ceux testés.

### Modele 1 : Regression Linéraire

In [72]:
modele_lineraire = LinearRegression()

# Validation croisée ( 5 folds, 3 métriques de perfromance, et récupération des scores d'entrainnement)
resultats = cross_validate(
    modele_lineraire, X_train, y_train_log,
    cv=5,
    scoring=["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"],
    return_train_score=True
)

print("Résultats de la valisation croisée sur les 5 fold d'entrainnement")

# le R2
R2_RegLin_CrossVal = resultats["test_r2"].mean().round(3)  
print("R² Cross_validate moyen :", R2_RegLin_CrossVal)

# Le RMSE Root Mean Squared Error (écart quadratique moyen))
# Il prend l'unité de la variable cible
RMSE_RegLin_CrossVal = (-resultats["test_neg_root_mean_squared_error"]).mean().round(3)
print("RMSE Cross_validate moyen :", RMSE_RegLin_CrossVal)

# Le MAE Mean Absolute Error (erreur absolue moyenne)
MAE_RegLin_CrossVal = (-resultats["test_neg_mean_absolute_error"]).mean().round(3)
print("MAE Cross_validate moyen :", MAE_RegLin_CrossVal)

# Temps en secondes pour chaque fold d'entrainnement 
print("Temps d'entraînement :", resultats['fit_time'])


Résultats de la valisation croisée sur les 5 fold d'entrainnement
R² Cross_validate moyen : 0.374
RMSE Cross_validate moyen : 1.027
MAE Cross_validate moyen : 0.761
Temps d'entraînement : [0.00323606 0.00257397 0.00187612 0.00124002 0.00103021]


/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overfl

#### Entrainnement du modele 
Affichage des Ecarts d'entrainnement et de la validation.   
(idéalement ils doivent etre proches)

In [74]:
modele_lineraire.fit(X_train, y_train_log)


print("R² train :", round(modele_lineraire.score(X_train, y_train_log), 3))
print("R² test :", round(modele_lineraire.score(X_test, y_test_log), 3))

R² train : 0.431
R² test : 0.413


/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overfl

#### Prédiction du modele

In [76]:
y_pred_log_lin = modele_lineraire.predict(X_test)

R2_RegLin_pred = round(r2_score(y_test_log, y_pred_log_lin), 3)
RMSE_RegLin_pred = round(root_mean_squared_error(y_test_log, y_pred_log_lin), 3)
MAE_RegLin_pred = round(mean_absolute_error(y_test_log, y_pred_log_lin), 3)

print("R² :", R2_RegLin_pred)
print("RMSE :", RMSE_RegLin_pred)
print("MAE :", MAE_RegLin_pred)


R² : 0.413
RMSE : 1.007
MAE : 0.767


/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/david/Documents/Formation AI Engineer OC/Projet_3/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


#### Affichage des scores Regression Linéraire

In [79]:
df_scores_RL = pd.DataFrame({
    "R2_regLin": [R2_RegLin_CrossVal, R2_RegLin_pred],
    "RMSE_regLin": [RMSE_RegLin_CrossVal, RMSE_RegLin_pred],
    "MAE_regLin": [MAE_RegLin_CrossVal, MAE_RegLin_pred],
}, index=["Cross-val", "Pedict"])

df_scores_RL

,R2_regLin,RMSE_regLin,MAE_regLin
Cross-val,0.374,1.027,0.761
Pedict,0.413,1.007,0.767


Les scores R2 RMSE et MAE sont tous rappochés.   
Ce qui signifie que l'entrainnement et la prédiction sont bien faites pas le modele.   
* Pour le R2, nous sommes un peu au dessus de ce qu'on peut avoir avec la moyenne. 
* Les Résutlats RMSE sont très elevés , le modele de Regression Linéraire est trop simple (Underfitting)

#### Interpretation des RMSE score 
Pour comproendre les RMSE , il faut savoir plusieurs choses :  
Le RMSE prend la meme echelle que les données à prédire (SiteEnrergyuse (kBtu)).   
Pour savoir si c'est proche de 0 ou elevé , il faut en calculer le pourcentage :     
*  Resultat =  (RMSE predict / Moyenne variable "SiteEnergyUse (kBtu)") *100.   
    

Savoir si le modele est stable :
 <table border="1" style="width: 100%; border-collapse: collapse;">
        <thead>
          <tr>
            <th style="padding: 8px; text-align: left;">Cas de figure</th>
            <th style="padding: 8px; text-align: left;">Interpretation</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td style="padding: 6px;">Resultat proche de 0</td>
            <td style="padding: 6px;">Le modele prédit bien et s'adapte aux nouvelles données</td>
          </tr>
          <tr>
            <td style="padding: 6px;">RMSE crossVal proche de 0 mais RMSE predict élevé</td>
            <td style="padding: 6px;">OVERFITTING (surapprentissage) Le modele a appris par coeur les données d'entrainnement mais ne sais pas prédire les nouvelles données</td>
          </tr>
          <tr>
            <td style="padding: 6px;">RMSE tous les 2 élevés</td>
            <td style="padding: 6px;"> UNDERFITTING (Sous-Apprentissage) Le modele  </td>
          </tr>
        </tbody>
</table>


#### Affichage du pourcentage d'erreur du modele

In [93]:
# Calcul de la moyenne sur la variable cible
moyenne_cible = building_consumption_ready_to_fe['SiteEnergyUse(kBtu)'].mean()

y_pred_lin = np.expm1(y_pred_log_lin)


erreur_relative = (y_pred_lin - y_test).abs() / y_test
print(erreur_relative.median() * 100)

53.72175814398401


La Regression Linéraire se trompe beaucoup trop et n'est probablement pas le modele adapté a ce dataset

### Modele 2 : Random Forest

In [102]:
# Modele a base d'arbres de décision
modele_randomForest = RandomForestRegressor(random_state=42, n_jobs=-1)

# Validation croisée
resultats = cross_validate(
    modele_randomForest, X_train, y_train_log,
    cv=5,
    scoring=["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"],
)

R2_RandForest_CrossVal = round(resultats["test_r2"].mean(), 3)
RMSE_RandForest_CrossVal = round(-resultats["test_neg_root_mean_squared_error"].mean(), 3)
MAE_RandForest_CrossVal = round(-resultats["test_neg_mean_absolute_error"].mean(), 3)

print("R² Cross_validate moyen :", R2_RandForest_CrossVal)
print("RMSE Cross_validate moyen :", RMSE_RandForest_CrossVal)
print("MAE Cross_validate moyen :", MAE_RandForest_CrossVal)


R² Cross_validate moyen : 0.546
RMSE Cross_validate moyen : 0.878
MAE Cross_validate moyen : 0.653


#### Entrainnement du modele 

In [ ]:
modele_randomForest.fit(X_train, y_train_log)

print("R² train :", round(modele_randomForest.score(X_train, y_train_log), 3))
print("R² test :", round(modele_randomForest.score(X_test, y_test_log), 3))

R² train : 0.938
R² test : 0.565


#### Prédiction du modele

In [97]:
y_pred_log_randForest = modele_randomForest.predict(X_test)

R2_RandForest_Pred = round(r2_score(y_test_log, y_pred_log_randForest), 3)
RMSE_RandForest_Pred = round(root_mean_squared_error(y_test_log, y_pred_log_randForest), 3)
MAE_RandForest_Pred = round(mean_absolute_error(y_test_log, y_pred_log_randForest), 3)

print("R² :", R2_RandForest_Pred)
print("RMSE :", RMSE_RandForest_Pred)
print("MAE :", MAE_RandForest_Pred)

R² : 0.565
RMSE : 0.867
MAE : 0.645


#### Affichage des scores Random Forest

In [86]:
df_scores_RL = pd.DataFrame({
    "R2": [R2_RandForest_CrossVal, R2_RandForest_Pred],
    "RMSE": [RMSE_RandForest_CrossVal, RMSE_RandForest_Pred],
    "MAE": [MAE_RandForest_CrossVal, MAE_RandForest_Pred],
}, index=["Cross-val", "Pedict"])

df_scores_RL

,R2,RMSE,MAE
Cross-val,0.546,0.878,0.653
Pedict,0.565,0.867,0.645


#### Affichage du pourcentage d'erreur du modele

In [92]:
y_pred_randForest = np.expm1(y_pred_log_randForest)

erreur_relative = (y_pred_randForest - y_test).abs() / y_test
print(erreur_relative.median() * 100)

46.82034188499239


Le modele Random Forest fait legèrement mieux que la regression linéraire mais c'est toujours insuffisant

### Modele 3 : SVR (Support Vector Regression)

In [106]:
modele_svr = TransformedTargetRegressor(
    regressor=SVR(kernel="rbf"),
    transformer=StandardScaler(),
)

# 1. Validation croisée sur le train
resultats = cross_validate(
    modele_svr, X_train, y_train_log,
    cv=5,
    scoring=["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"],
)

R2_svr_CrossVal = round(resultats["test_r2"].mean(), 3)
RMSE_svr_CrossVal = round(-resultats["test_neg_root_mean_squared_error"].mean(), 3)
MAE_svr_CrossVal = round(-resultats["test_neg_mean_absolute_error"].mean(), 3)

print("R² Cross_validate moyen :", R2_svr_CrossVal)
print("RMSE Cross_validate moyen :", RMSE_svr_CrossVal)
print("MAE Cross_validate moyen :", MAE_svr_CrossVal)

R² Cross_validate moyen : 0.47
RMSE Cross_validate moyen : 0.949
MAE Cross_validate moyen : 0.718


#### Entrinnement du modele 

In [107]:
modele_svr.fit(X_train, y_train_log)

print("R² train :", round(modele_svr.score(X_train, y_train_log), 3))
print("R² test :", round(modele_svr.score(X_test, y_test_log), 3))

R² train : 0.524
R² test : 0.516


#### Prediction du modele

In [108]:
y_pred_log = modele_svr.predict(X_test)

R2_svr_pred = round(r2_score(y_test_log, y_pred_log), 3)
RMSE_svr_pred = round(root_mean_squared_error(y_test_log, y_pred_log), 3)
MAE_svr_pred = round(mean_absolute_error(y_test_log, y_pred_log), 3)

print("R² :", R2_svr_pred)
print("RMSE :", RMSE_svr_pred)
print("MAE :", MAE_svr_pred)

R² : 0.516
RMSE : 0.914
MAE : 0.689


#### Affichage des scores SVR

In [109]:
df_scores_SVR = pd.DataFrame({
    "R2": [R2_svr_CrossVal, R2_svr_pred],
    "RMSE": [RMSE_svr_CrossVal, RMSE_svr_pred],
    "MAE": [MAE_svr_CrossVal, MAE_svr_pred],
}, index=["Cross-val", "Pedict"])

df_scores_SVR

,R2,RMSE,MAE
Cross-val,0.470,0.949,0.718
Pedict,0.516,0.914,0.689


#### Affichage du pourcentage d'erreur sur le modele

In [111]:
# Calcul de la moyenne sur la variable cible
moyenne_cible = building_consumption_ready_to_fe['SiteEnergyUse(kBtu)'].mean()


y_pred_svr = np.expm1(y_pred_log)


erreur_relative = (y_pred_lin - y_test).abs() / y_test
print(erreur_relative.median() * 100)

53.72175814398401


Les scores sont beacoup plus realistes avec ce modele

### Compraison des 3 modeles et Définition du modele a optimiser

Pour la regression Linéaire et la Random Forest , mon premier test a été de faire avec les valeurs cible classique.    
Ensuite, j'ai refait les RL et RandForest en transformant la variable cible avec le logarithme.   

In [ ]:
# ici faire un DF avec tous les résultats des 3 modeles
# definir le meilleur et l'optimiser 

### Optimisation et interprétation du modèle

A réaliser :
* Reprennez le meilleur algorithme que vous avez sécurisé via l'étape précédente, et réalisez une GridSearch de petite taille sur au moins 3 hyperparamètres.
* Si le meilleur modèle fait partie de la famille des modèles à arbres (RandomForest, GradientBoosting) alors utilisez la fonctionnalité feature importance pour identifier les features les plus impactantes sur la performance du modèle. Sinon, utilisez la méthode Permutation Importance de sklearn.

L'importance par permutation (Permutation Feature Importance)

UNIQUEMENT arbre de décision : La diminution moyenne d'impureté (MDI / Mean Decrease Impurity)



In [ ]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE